In [ ]:
import pandas as pd
import sys
sys.path.append('../..')
from src.sawmill.sawmill import Sawmill
import matplotlib.pyplot as plt
import numpy as np
from src.sawmill.causal_discoverer import CausalDiscoverer
from src.sawmill.ate import ATECalculator
from src.sawmill.tag_utils import TagUtils
import networkx as nx

In [ ]:



sawmill_raw_ates = [-156.47, -0.68, 0.01, 1.10, 257.47, 112.66, 27.28, 258.64, 121.45, 33.98, 
                    258.57, 85.66, 42.64, 2.00, 2.11, 1.97, 1.96, 1.6, 0.87, 1.78, 0.62, 0.12]


ground_truth_ates = [sawmill_raw_ates[0],sawmill_raw_ates[1],sawmill_raw_ates[2],sawmill_raw_ates[3],258.43, 114.86, 28.71, 258.43, 114.86, 28.71, 258.43, 114.86, 28.71, 2, 2, 2, 2, 2, 2, 2, 2, 2]


regression_raw_ates = [0] * 22

gpt_times = [0]*22
gpt_mrrs = [0]*22
gpt_raw_ates = [0]*22


In [ ]:
sawmill_mrrs = [(1/2 + 1/5 + 1)/3, 
                1,1,1,
                1,1,1,1,1,1,1,1,1,
                (1+1/2+1/2)/3, (1+1/2+1/3)/3, (1+1/2+1/2)/3,
                (1+1/2+1/2)/3, (1+1/2+1/2)/3, (1/2+1/3+1/3)/3,
                (1+1/2+1/2)/3,(1+1/2+1/2)/3,(1+1/2+1/2)/3]

regression_mrrs = [0] * 22

---

# PostgreSQL

In [ ]:
filename = '../../datasets_raw/tpc-ds/parameter_sweep_1_filtered.log'

s_postgresql = Sawmill(
    filename=filename,
    workdir="../../datasets/tpc-ds/",
)
parse_time = s_postgresql.parse(regex_dict={"Date": r'\d{4}-\d{2}-\d{2}',
        "Time": r'\d{2}:\d{2}:\d{2}\.\d{3}(?= EST \[ )', 
        "sessionID" : r'(?<=EST \[ )\S+\.\S+',
        "tID": r'3/\d+(?= ] )'
        },message_prefix=r'\d{4}-\d{2}-\d{2} \d{2}:\d{2}:\d{2}.\d{3}')
s_postgresql.set_causal_unit("sessionID")
s_postgresql.prepare(count_occurences=True, custom_agg={'sessionID': ['mode']}, reject_prunable_edges=False)

In [ ]:
# Regression
outcome_tag = 'duration mean'
causes_tags = ['work_mem mean', 'max_parallel_workers mean']

df, _ = s_postgresql.explore_candidate_causes(outcome_tag, skip_lasso=True, use_multivariate_regression=True, lasso_alpha=0.1)
regression_raw_ates[0] = 0
print(regression_raw_ates[0])

df_for_rank = df[df['P-value'] < 0.05].reset_index(drop=True)
causes_ranks = [df_for_rank[df_for_rank['Candidate Tag'] == c].index[0] + 1 if any(df_for_rank['Candidate Tag'] == c) else 0 for c in causes_tags]


df2, _ = s_postgresql.explore_candidate_causes(causes_tags[0], skip_lasso=True, use_multivariate_regression=True, lasso_alpha=0.1)
df2_for_rank = df2[df2['P-value'] < 0.05].reset_index(drop=True)
causes_ranks.append((df2_for_rank[df2_for_rank['Candidate Tag'] == causes_tags[1]].index[0] + 1 if any(df2_for_rank['Candidate Tag'] == causes_tags[1]) else 0))


causes_reciprocal_ranks = [1/j if j != 0 else 0 for j in causes_ranks]
mrr = sum(causes_reciprocal_ranks) / len(causes_reciprocal_ranks)
print(causes_ranks)
print(mrr)
regression_mrrs[0] = mrr

In [ ]:
gptoutput,time_elapsed = CausalDiscoverer.gpt_baseline(s_postgresql.prepared_log, 'baseline-postresql-conf','max_parallel_workers mean', vars_df=s_postgresql.prepared_variables)


In [ ]:
# GPT
outcome_tag = 'duration mean'
causes_tags = ['work_mem mean', 'max_parallel_workers mean']

gptoutput,time_elapsed = CausalDiscoverer.gpt_baseline(s_postgresql.prepared_log, 'baseline-postresql', outcome_tag, vars_df=s_postgresql.prepared_variables)

with open(gptoutput, 'r') as f:
    lines = f.readlines()
    lines = lines[lines.index('----------------\n')+1:]

    causes = [line.split('.')[1].strip() for line in lines if line[0].isdigit() and (line[1] == '.' or line[2] == '.')]
    dag = [[TagUtils.name_of(s_postgresql.prepared_variables, l.strip().strip('-').strip(), 'prepared') for l in line.split('->')] for line in lines if '->' in line]
    
    # for each element in causes_tags, find the rank in causes
    causes_ranks = [causes.index(c) + 1 if c in causes else 0 for c in causes_tags]
    causes_reciprocal_ranks = [1/j if j != 0 else 0 for j in causes_ranks]
    print(causes_ranks)

    # Create the dag and calculate the ATE
    graph = nx.DiGraph()
    graph.add_edges_from(dag)

    ate = ATECalculator.get_ate_and_confidence(s_postgresql.prepared_log, s_postgresql.prepared_variables, causes_tags[0], outcome_tag, None, graph)

    gpt_mrrs[0] = sum(causes_reciprocal_ranks) / len(causes_reciprocal_ranks)
    print(f'MRR: {gpt_mrrs[0]}')
    gpt_raw_ates[0] = ate['ATE']
    print(f'ATE: {gpt_raw_ates[0]}')
    gpt_times[0] = time_elapsed
    print(f'Elapsed time: {gpt_times[0]}')   

---

# OpenStack

## Cinder

In [ ]:
# Cinder
s_cinder = Sawmill("../../datasets_raw/OpenStack/Cinder_combined_all.log", workdir="../../datasets/OpenStack/Cinder/")
s_cinder.parse(regex_dict={"ID" : r'Test_\d+_round_\d',
                    "Date": r'\d{4}-\d{2}-\d{2} \d{2}:\d{2}:\d{2}.\d{6}'},
                    message_prefix=r'Test_\d+_round_\d \d{4}-\d{2}-\d{2} \d{2}:\d{2}:\d{2}.\d{6}',
                    enable_gpt_tagging=True)
s_cinder.set_causal_unit("ID")
imp_dict = {k:'zero_imp' for k in s_cinder.parsed_variables['Name'].values}
s_cinder.prepare(count_occurences=True, custom_agg={'ID':['mode']}, custom_imp=imp_dict, lasso_alpha=0.1, reject_prunable_edges=False)

In [ ]:
# Regression
outcome_tag = 'TemplateId=cd2639ad+sum'
causes_tags = ['8acf41e5_12 last attached']

df, _ = s_cinder.explore_candidate_causes(outcome_tag, skip_lasso=True, use_multivariate_regression=True, lasso_alpha=0.1)
regression_raw_ates[1] = (df[df['Candidate Tag'] == causes_tags[0]]['Slope'].values[0])
print(regression_raw_ates[1])

causes_ranks = [df[df['Candidate Tag'] == c].index[0] + 1 if any(df['Candidate Tag'] == c) else 0 for c in causes_tags]
causes_reciprocal_ranks = [1/j if j != 0 else 0 for j in causes_ranks]
mrr = sum(causes_reciprocal_ranks) / len(causes_reciprocal_ranks)
print(causes_ranks)
print(mrr)
regression_mrrs[1] = mrr

In [ ]:
# GPT
outcome_tag = 'TemplateId=cd2639ad+sum'
causes_tags = ['8acf41e5_12 last attached']

gptoutput,time_elapsed = CausalDiscoverer.gpt_baseline(s_cinder.prepared_log, 'baseline-cinder', outcome_tag, vars_df=s_cinder.prepared_variables)

with open(gptoutput, 'r') as f:
    lines = f.readlines()
    lines = lines[lines.index('----------------\n')+1:]

    causes = [line.split('.')[1].strip() for line in lines if line[0].isdigit() and (line[1] == '.' or line[2] == '.')]
    dag = [[TagUtils.name_of(s_postgresql.prepared_variables, l.strip().strip('-').strip(), 'prepared') for l in line.split('->')] for line in lines if '->' in line]
    
    # for each element in causes_tags, find the rank in causes
    causes_ranks = [causes.index(c) + 1 if c in causes else 0 for c in causes_tags]
    causes_reciprocal_ranks = [1/j if j != 0 else 0 for j in causes_ranks]
    print(causes_ranks)

    # Create the dag and calculate the ATE
    graph = nx.DiGraph()
    graph.add_edges_from(dag)

    ate = ATECalculator.get_ate_and_confidence(s_cinder.prepared_log, s_cinder.prepared_variables, causes_tags[0], outcome_tag, None, graph)

    index = 1
    gpt_mrrs[index] = sum(causes_reciprocal_ranks) / len(causes_reciprocal_ranks)
    print(f'MRR: {gpt_mrrs[index]}')
    gpt_raw_ates[index] = ate['ATE']
    print(f'ATE: {gpt_raw_ates[index]}')
    gpt_times[index] = time_elapsed
    print(f'Elapsed time: {gpt_times[index]}')   

## Neutron

In [ ]:
s_neutron = Sawmill("../../datasets/OpenStack/Neutron_combined_all.log", workdir="../../datasets/OpenStack/Neutron/")
s_neutron.parse(regex_dict={"ID" : r'Test_\d+_round_\d',
                    "DateTime": r'\d{4}-\d{2}-\d{2} \d{2}:\d{2}:\d{2}.\d{6}'},
                    message_prefix=r'Test_\d+_round_\d \d{4}-\d{2}-\d{2} \d{2}:\d{2}:\d{2}.\d{6}', enable_gpt_tagging=True)
s_neutron.set_causal_unit("ID")
imp_dict = {k:'zero_imp' for k in s_neutron.parsed_variables['Name'].values}
s_neutron.prepare(count_occurences=True, custom_agg={'ID':['mode']},custom_imp=imp_dict, lasso_alpha=0.1, reject_prunable_edges=False)

In [ ]:
# Regression
outcome_tag = 'TemplateId=cd2639ad+sum'
causes_tags = ['connection_result mean']

df, _ = s_neutron.explore_candidate_causes(outcome_tag, skip_lasso=True, use_multivariate_regression=True, lasso_alpha=0.1)
values = df[df['Candidate Tag'] == causes_tags[0]]['Slope'].values
regression_raw_ates[2] = values[0] if len(values) > 0 else 0
print(regression_raw_ates[2])

causes_ranks = [df[df['Candidate Tag'] == c].index[0] + 1 if any(df['Candidate Tag'] == c) else 0 for c in causes_tags]
causes_reciprocal_ranks = [1/j if j != 0 else 0 for j in causes_ranks]
mrr = sum(causes_reciprocal_ranks) / len(causes_reciprocal_ranks)
print(causes_ranks)
print(mrr)
regression_mrrs[2] = mrr

In [ ]:
# GPT

outcome_tag = 'TemplateId=cd2639ad+sum'
causes_tags = ['connection_result mean']

gptoutput,time_elapsed = CausalDiscoverer.gpt_baseline(s_neutron.prepared_log, 'baseline-neutron', outcome_tag, vars_df=s_neutron.prepared_variables)



## Nova

In [ ]:
s_nova = Sawmill("../../datasets/OpenStack/Nova_combined_all.log", workdir="../../datasets/OpenStack/Nova/")
s_nova.parse(regex_dict={"ID" : r'Test_\d+_round_\d',
                    "DateTime": r'\d{4}-\d{2}-\d{2} \d{2}:\d{2}:\d{2}.\d{6}'},
                    message_prefix=r'Test_\d+_round_\d \d{4}-\d{2}-\d{2} \d{2}:\d{2}:\d{2}.\d{6}', enable_gpt_tagging=True)
s_nova.set_causal_unit("ID")
imp_dict = {k:'zero_imp' for k in s_nova.parsed_variables['Name'].values}
s_nova.prepare(count_occurences=True, custom_agg={'ID':['mode']},custom_imp=imp_dict, lasso_alpha=0.1, reject_prunable_edges=False)

In [ ]:
# Regression
outcome_tag = 'TemplateId=cd2639ad+sum'
causes_tags = ['RETRY_COUNT last #4']

df, _ = s_nova.explore_candidate_causes(outcome_tag, skip_lasso=True, use_multivariate_regression=True, lasso_alpha=0.1)
values = df[df['Candidate Tag'] == causes_tags[0]]['Slope'].values
regression_raw_ates[3] = values[0] if len(values) > 0 else 0
print(regression_raw_ates[3])

causes_ranks = [df[df['Candidate Tag'] == c].index[0] + 1 if any(df['Candidate Tag'] == c) else 0 for c in causes_tags]
causes_reciprocal_ranks = [1/j if j != 0 else 0 for j in causes_ranks]
mrr = sum(causes_reciprocal_ranks) / len(causes_reciprocal_ranks)
print(causes_ranks)
print(mrr)
regression_mrrs[3] = mrr

In [ ]:
# GPT
outcome_tag = 'TemplateId=cd2639ad+sum'
causes_tags = ['RETRY_COUNT last #4']

gptoutput,time_elapsed = CausalDiscoverer.gpt_baseline(s_nova.prepared_log, 'baseline-nova', outcome_tag, vars_df=s_nova.prepared_variables)



---

# Proprietary

In [ ]:
total_users = 1000
faulty_scenaria = [
    int(0.5 * total_users),
    int(0.1 * total_users),
    int(0.01 * total_users),
]
prob_scenaria = [[100, 10], 
                    [50, 10], 
                    [20, 10]
                    ] 

k = 4
 
for i in faulty_scenaria:
    for j in prob_scenaria:
        args = {"users": total_users, "faulty_users": i, "fail_prob_pct": j}

        
        filename = f'../../datasets_raw/proprietary_logs/proprietary_{args["users"]}users_{args["faulty_users"]}faulty_{args["fail_prob_pct"][0]}pctfailfaulty_{args["fail_prob_pct"][1]}pctfailnormal'
        orig_filename = f"../../datasets_raw/proprietary_logs/proprietary_original.log"

        s_prop = Sawmill(
            filename + ".log",
            workdir="../../datasets/proprietary_logs/proprietaryprietary_eval",
        )
        
        parse_time = s_prop.parse(
            regex_dict=(
                s_prop.DEFAULT_REGEX_DICT
                | {
                    "UnixTimestamp": r"16\d{11}(?=\sINFO|\sWARN|\sERROR)",
                    "User": r"user_\d+",
                }
            ),
            sim_thresh=0.9,
            enable_gpt_tagging=True
        )
        s_prop.set_causal_unit("User")
        prep_time = s_prop.prepare(custom_agg={'User':['mode']}, reject_prunable_edges=False)
        #gptoutput,time_elapsed = CausalDiscoverer.gpt_baseline(s_prop.prepared_log, f'baseline-prop-{i}-{j[0]}', '73b16c0a_196 mean', vars_df=s_prop.prepared_variables)
        #continue
        c, cand_time = s_prop.explore_candidate_causes("73b16c0a_196 mean", lasso_alpha=0.1)
        print(c.loc[0])

        df, _ = s_prop.explore_candidate_causes("73b16c0a_196 mean", skip_lasso=True, use_multivariate_regression=True, lasso_alpha=0.1)
        values = df[df['Candidate Tag'] == 'version mean']['Slope'].values
        regression_raw_ates[k] = values[0] if len(values) > 0 else 0
        print(regression_raw_ates[k])

        df_for_rank = df[df['P-value'] < 0.05].reset_index(drop=True)
        causes_ranks = [df_for_rank[df_for_rank['Candidate Tag'] == 'version mean'].index[0] + 1 if any(df_for_rank['Candidate Tag'] == 'version mean') else 0]
        causes_reciprocal_ranks = [1/j if j != 0 else 0 for j in causes_ranks]
        mrr = sum(causes_reciprocal_ranks) / len(causes_reciprocal_ranks)
        print(causes_ranks)
        print(mrr)
        regression_mrrs[k] = mrr
        k = k + 1


---

# XYZ

In [ ]:
import os

files = os.listdir("~/causal-log/datasets_raw/xyz_extended")
files.sort()

k = 13

for i in files:
    if i.endswith(".log"):
        filename = "~/causal-log/datasets_raw/xyz_extended/" + i

        with open(filename.split(".")[0] + ".json", "r") as fjson:
            l = fjson.readlines()
            num_total_variables = int(l[3].split(":")[1].strip().strip(","))
            noise_radius = int(l[4].split(":")[1].strip())

        

        # Parse and prepare the log file
        s_xyz = Sawmill(
            filename=filename,
            workdir="../../datasets/xyz_extended/",
        )
        parse_time = s_xyz.parse(
            regex_dict={
                "timestamp": r"\d{4}-\d{2}-\d{2}T\d{2}:\d{2}:\d{2}\.\d{6}Z",
                "machine": r"machine_\d+",
            }
        )
        s_xyz.set_causal_unit("machine")
        prep_time = s_xyz.prepare(count_occurences=True, reject_prunable_edges=False)

        #s_xyz.accept('x mean', 'y mean')
        #s_xyz.accept('z mean', 'y mean')
        #val = s_xyz.get_adjusted_ate('x mean', 'y mean')
        #print(f'num_total_variables is {num_total_variables}')
        #print(f'noise_radius is {noise_radius}')
        #print(f"ATE is {val}")
        #continue

        if False:
            gptoutput, time_elapsed = CausalDiscoverer.gpt_baseline(
                s_xyz.prepared_log,
                f"baseline-xyz-conf-{num_total_variables}-{noise_radius}",
                "x mean",
                vars_df=s_xyz.prepared_variables,
                k=9,
            )

        df, _ = s_xyz.explore_candidate_causes(
            "y mean", skip_lasso=True, use_multivariate_regression=True, lasso_alpha=0.1
        )
        values = df[df["Candidate Tag"] == "x mean"]["Slope"].values
        regression_raw_ates[k] = values[0] if len(values) > 0 else 0
        print(regression_raw_ates[k])

        causes_tags = ["x mean", "z mean"]
        df_for_rank = df[df['P-value'] < 0.05].reset_index(drop=True)
        causes_ranks = [
            df_for_rank[df_for_rank["Candidate Tag"] == c].index[0] + 1
            if any(df_for_rank["Candidate Tag"] == c)
            else 0
            for c in causes_tags
        ]

        df2, _ = s_xyz.explore_candidate_causes(
            "x mean", skip_lasso=True, use_multivariate_regression=True, lasso_alpha=0.1
        )
        df2_for_rank = df2[df2['P-value'] < 0.05].reset_index(drop=True)
        causes_ranks.append(
            (
                df2_for_rank[df2_for_rank["Candidate Tag"] == "z mean"].index[0] + 1
                if any(df2_for_rank["Candidate Tag"] == "z mean")
                else 0
            )
        )

        causes_reciprocal_ranks = [1 / j if j != 0 else 0 for j in causes_ranks]
        mrr = sum(causes_reciprocal_ranks) / len(causes_reciprocal_ranks)
        print(causes_ranks)
        print(mrr)
        regression_mrrs[k] = mrr

        k = k + 1

---

In [ ]:
acc_df = pd.DataFrame({'true ATE': ground_truth_ates, 'sawmill ATE':sawmill_raw_ates, 'regression ATE': regression_raw_ates,
                       'sawmill MRR': sawmill_mrrs, 'regression MRR': regression_mrrs})
acc_df['Sawmill ATE Error'] = abs((acc_df['sawmill ATE'] - acc_df['true ATE'])/acc_df['true ATE'])
acc_df['regression ATE Error'] = abs((acc_df['regression ATE'] - acc_df['true ATE'])/acc_df['true ATE'])
acc_df.to_csv('accuracy.csv')
acc_df

In [ ]:
acc_df

In [ ]:
acc_df.loc[13:22, 'regression ATE Error'].mean()

In [ ]:
%env PATH=/usr/local/texlive/2023/bin/x86_64-linux:~/logs-venv/bin:~/.vscode-server/bin/0ee08df0cf4527e40edc9aa28f4b5bd38bbff2b2/bin/remote-cli:~/logs-venv/bin:~/.pyenv/shims:~/.local/bin:~/.cargo/bin:~/apache-maven-3.6.3/bin:/ssd1/geoffxy/opt/Python-2.7.17:~/LearnedLSM/ycsb-0.17.0/bin:/usr/local/sbin:/usr/local/bin:/usr/bin:/opt/cuda/bin:/opt/cuda/nsight_compute:/opt/cuda/nsight_systems/bin:/usr/lib/jvm/default/bin:/usr/bin/site_perl:/usr/bin/vendor_perl:/usr/bin/core_perl:/usr/lib/rustup/bin

In [ ]:
!echo $PATH

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt

acc_df = pd.read_csv('accuracy.csv')

In [ ]:
acc_df

# Plot

In [ ]:
from mpl_toolkits.axes_grid1 import make_axes_locatable
import numpy as np


groups = ['PostgreSQL', 'OpenStack\nCinder', 'OpenStack\nNeutron', 'OpenStack\nNova', 
            'Proprietary\nF=0.5, p_f=1.0',
            'Proprietary\nF=0.5, p_f=0.5',
            'Proprietary\nF=0.5, p_f=0.2',
            'Proprietary\nF=0.1, p_f=1.0',
            'Proprietary\nF=0.1, p_f=0.5',
            'Proprietary\nF=0.1, p_f=0.2',
            'Proprietary\nF=0.01, p_f=1.0',
            'Proprietary\nF=0.01, p_f=0.5',
            'Proprietary\nF=0.01, p_f=0.2',
            'XYZ\nV=10, R=1',
            'XYZ\nV=10, R=5',
            'XYZ\nV=10, R=10',
            'XYZ\nV=100, R=1',
            'XYZ\nV=100, R=5',
            'XYZ\nV=100, R=10',
            'XYZ\nV=1000, R=1',
            'XYZ\nV=1000, R=5',
            'XYZ\nV=1000, R=10',
          
          ]
bar1 = acc_df['sawmill ATE'] / acc_df['true ATE']
bar2 = acc_df['regression ATE'] / acc_df['true ATE']
# Setting the positions and height for the bars
height = 0.3 
colors = ["#7F9ABA",  "#7FBA82", "#BA7FB7",  ]


# Plotting
fig, (ax1, ax2) = plt.subplots(2, figsize=(10,20), height_ratios=[4/22, 18/22])
plt.rcParams['font.size'] = 20



pos1 = np.arange(4)
ax1.barh(pos1 - height/2, bar1[:4], height, label='Sawmill', color = colors[0])
ax1.barh(pos1 + height/2, bar2[:4], height, label='Regression', color = colors[1])
ax1.set_ylim(-height*2, 3+2*height)
ax1.axvline(x=1, ymin=-height*2, ymax=3+2*height, color='black', linestyle='--', linewidth=0.8)
ax1.axvline(x=0, ymin=-height*2, ymax=3+2*height, color='black', linestyle='-', linewidth=0.8)


ax1.set_xlabel('Recovered ATE relative to Sawmill')
ax1.set_yticks(pos1, groups[:4], fontsize=20)


pos2 = np.arange(18)
divider = make_axes_locatable(ax2)
ax2.barh(pos2 - height/2, bar1[4:], height, label='Sawmill', color = colors[0])
ax2.barh(pos2 + height/2, bar2[4:], height, label='Regression', color = colors[1])
ax2.set_xlim(-153, -152.25)
ax2.spines['right'].set_visible(False)
ax2.set_xticks([-152.66])

ax_ext = divider.new_horizontal(size="400%", pad=0.1)
fig.add_axes(ax_ext)
ax_ext.barh(pos2 - height/2, bar1[4:], height, label='Sawmill', color = colors[0])
ax_ext.barh(pos2 + height/2, bar2[4:], height, label='Regression', color = colors[1])
ax_ext.set_xlim(-0.5, 2.5)
ax_ext.tick_params(left=False, labelleft=False)
ax_ext.spines['left'].set_visible(False)
ax_ext.set_xticks([m for m in range(0,3,1)])

ax_ext.set_xlabel('Recovered ATE relative to ground truth')
ax2.set_yticks(pos2, groups[4:])
ax2.set_ylim(-height*2, 17+2*height)
ax_ext.set_ylim(-height*2, 17+2*height)
ax_ext.axvline(x=1, ymin=-height*2, ymax=17+2*height, color='black', linestyle='--',  linewidth=0.8)
ax_ext.axvline(x=0, ymin=-height*2, ymax=3+2*height, color='black', linestyle='-', linewidth=0.8)

d = .7  # proportion of horizontal to vertical extent of the slanted line
kwargs = dict(marker=[(-d, -1), (d, 1)], markersize=12,
              linestyle="none", color='k', mec='k', mew=1, clip_on=False)
ax2.plot([1, 1], [1, 0], transform=ax2.transAxes, **kwargs)
ax_ext.plot([0, 0], [0, 1], transform=ax_ext.transAxes, **kwargs)


# Flip y axes
ax1.invert_yaxis()
ax2.invert_yaxis()
ax_ext.invert_yaxis()


# Adding legend
ax_ext.legend(loc='upper right')

# Displaying the plot
plt.subplots_adjust(wspace=0, hspace=0.2)
plt.tight_layout()
plt.show
plt.savefig('accuracy_ate.png', dpi=300)


In [ ]:
for i in range(len(sawmill_raw_ates)):
    print(f"{ground_truth_ates[i]:.2f}&{sawmill_raw_ates[i]:.2f}&{regression_raw_ates[i]:.2f}\\\\")

In [ ]:
from mpl_toolkits.axes_grid1 import make_axes_locatable


groups = ['PostgreSQL', 'OpenStack\nCinder', 'OpenStack\nNeutron', 'OpenStack\nNova', 
            'Proprietary\nF=0.5, p_f=1.0',
            'Proprietary\nF=0.5, p_f=0.5',
            'Proprietary\nF=0.5, p_f=0.2',
            'Proprietary\nF=0.1, p_f=1.0',
            'Proprietary\nF=0.1, p_f=0.5',
            'Proprietary\nF=0.1, p_f=0.2',
            'Proprietary\nF=0.01, p_f=1.0',
            'Proprietary\nF=0.01, p_f=0.5',
            'Proprietary\nF=0.01, p_f=0.2',
            'XYZ\nV=10, R=1',
            'XYZ\nV=10, R=5',
            'XYZ\nV=10, R=10',
            'XYZ\nV=100, R=1',
            'XYZ\nV=100, R=5',
            'XYZ\nV=100, R=10',
            'XYZ\nV=1000, R=1',
            'XYZ\nV=1000, R=5',
            'XYZ\nV=1000, R=10',
          
          ]

# Setting the positions and height for the bars
height = 0.3 
colors = ["#7F9ABA",  "#7FBA82", "#BA7FB7",  ]


# Plotting
fig, ax1 = plt.subplots(1, figsize=(10,20))
plt.rcParams['font.size'] = 20



pos1 = np.arange(22)
ax1.barh(pos1 - height/2, acc_df['sawmill MRR'], height, label='Sawmill', color = colors[0])
ax1.barh(pos1 + height/2, acc_df['regression MRR'], height, label='Regression', color = colors[1])
ax1.set_ylim(-2*height, 21+2*height)
ax1.axvline(x=1, ymin=-height*2, ymax=3+2*height, color='black', linestyle='--', linewidth=0.8)
ax1.axvline(x=0, ymin=-height*2, ymax=3+2*height, color='black', linestyle='-', linewidth=0.8)


ax1.set_xlabel('MRR')
ax1.set_yticks(pos1, groups, fontsize=20)



# Flip y axes
ax1.invert_yaxis()



# Adding legend
ax1.legend(loc='upper right')

# Displaying the plot
plt.tight_layout()
plt.show
plt.savefig('accuracy_mrr.png', dpi=300)


In [ ]:
for i in range(len(acc_df)):
    print(f"&{acc_df.loc[i, 'sawmill MRR']:.4f}&{acc_df.loc[i, 'regression MRR']:.4f}\\\\")

In [ ]:
acc_df['sawmill MRR'].mean()

In [ ]:
acc_df['regression MRR'].mean()

In [ ]:
acc_df['Sawmill ATE Error'][4:].mean()

In [ ]:
acc_df['regression ATE Error'][4:].mean()